In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TEST_CSV     = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'
MODEL_H5     = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'
MODEL_TFLITE = '/content/drive/MyDrive/isaret_dili_beyni_v2_android.tflite'

# ──────────────────────────────────────────────
# 1. VERİ YÜKLEME
# ──────────────────────────────────────────────
df = pd.read_csv(TEST_CSV)
print(f"Ham veri shape: {df.shape}")

META_COLS  = ['video_name', 'class_id', 'frame_no']
COORD_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Koordinat sütun sayısı: {len(COORD_COLS)}")

# ──────────────────────────────────────────────
# 2. MODEL GİRİŞ ŞEKLİNE GÖRE SEQUENCE OLUŞTURMA
# ──────────────────────────────────────────────
# Model (None, 60, 211) bekliyor
# CSV'de her satır = 1 frame
# video_name + class_id bazında grupla → her video bir sequence

TIMESTEPS = 60   # modelin beklediği
FEATURES  = 211  # modelin beklediği

# Eğitimde muhtemelen ilk 211 koordinat kullanılmış
# (225 özellik var ama model 211 istiyor — fazlayı at)
USE_COLS = COORD_COLS[:FEATURES]
print(f"Kullanılacak koordinat sütunu: {len(USE_COLS)}")

sequences = []
labels    = []

grouped = df.groupby(['video_name', 'class_id'], sort=False)
print(f"Toplam video (örnek) sayısı: {len(grouped)}")

for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    frames = group[USE_COLS].values.astype(np.float32)  # (n_frames, 211)
    n = len(frames)

    if n >= TIMESTEPS:
        # Ortadan TIMESTEPS kadar al
        start = (n - TIMESTEPS) // 2
        seq   = frames[start:start + TIMESTEPS]
    else:
        # Sona sıfır pad
        pad = np.zeros((TIMESTEPS - n, FEATURES), dtype=np.float32)
        seq = np.vstack([frames, pad])

    sequences.append(seq)
    labels.append(class_id)

X = np.array(sequences, dtype=np.float32)   # (n_videos, 60, 211)
y_raw = np.array(labels)

le          = LabelEncoder()
y           = le.fit_transform(y_raw)
class_names = le.classes_
NUM_CLASSES = len(class_names)

print(f"\nSequence array shape : {X.shape}")
print(f"Sınıf sayısı         : {NUM_CLASSES}")
print(f"Toplam örnek         : {len(y)}")

# ──────────────────────────────────────────────
# 3. H5 TAHMİN
# ──────────────────────────────────────────────
print("\n── H5 modeli yükleniyor ──")
model_h5 = tf.keras.models.load_model(MODEL_H5)
print(f"Model giriş şekli: {model_h5.input_shape}")

probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)
acc_h5    = accuracy_score(y, y_pred_h5)
print(f"H5 Doğruluğu: {acc_h5*100:.2f}%")

# ──────────────────────────────────────────────
# 4. TFLite TAHMİN
# ──────────────────────────────────────────────
print("\n── TFLite modeli yükleniyor ──")
interpreter = tf.lite.Interpreter(model_path=MODEL_TFLITE)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f"TFLite giriş: {input_details[0]['shape']}")

preds_tflite = []
for i in range(len(X)):
    inp = np.expand_dims(X[i], axis=0)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    preds_tflite.append(np.argmax(out))

y_pred_tflite = np.array(preds_tflite)
acc_tflite    = accuracy_score(y, y_pred_tflite)
print(f"TFLite Doğruluğu: {acc_tflite*100:.2f}%")

# ──────────────────────────────────────────────
# 5. SINIFLANDIRMA RAPORLARI
# ──────────────────────────────────────────────
print("\n══ H5 Sınıflandırma Raporu ══")
print(classification_report(y, y_pred_h5,
      target_names=[str(c) for c in class_names], zero_division=0))

print("\n══ TFLite Sınıflandırma Raporu ══")
print(classification_report(y, y_pred_tflite,
      target_names=[str(c) for c in class_names], zero_division=0))

# ──────────────────────────────────────────────
# 6. GÖRSEL ÇIKTILAR
# ──────────────────────────────────────────────

# 6a. Genel Metrik Karşılaştırma
metrics = {
    'Accuracy':   [acc_h5*100, acc_tflite*100],
    'F1 (macro)': [f1_score(y,y_pred_h5,     average='macro',zero_division=0)*100,
                   f1_score(y,y_pred_tflite,  average='macro',zero_division=0)*100],
    'Precision':  [precision_score(y,y_pred_h5,    average='macro',zero_division=0)*100,
                   precision_score(y,y_pred_tflite, average='macro',zero_division=0)*100],
    'Recall':     [recall_score(y,y_pred_h5,    average='macro',zero_division=0)*100,
                   recall_score(y,y_pred_tflite, average='macro',zero_division=0)*100],
}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics)); w = 0.35
bars1 = ax.bar(x-w/2, [v[0] for v in metrics.values()], w, label='H5',     color='#4C72B0')
bars2 = ax.bar(x+w/2, [v[1] for v in metrics.values()], w, label='TFLite', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(metrics.keys(), fontsize=12)
ax.set_ylim(0, 115); ax.set_ylabel('%'); ax.legend(fontsize=11)
ax.set_title('H5 vs TFLite — Genel Metrik Karşılaştırması', fontsize=14, fontweight='bold')
for b in list(bars1)+list(bars2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
            f'{b.get_height():.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/metric_comparison.png', dpi=150)
plt.show()

# 6b. Confusion Matrix H5
cm_h5_norm = confusion_matrix(y, y_pred_h5).astype(float)
cm_h5_norm /= cm_h5_norm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm_h5_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.2, annot=False)
ax.set_title('Confusion Matrix — H5 (Normalize)', fontsize=15, fontweight='bold')
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek')
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.tick_params(axis='y', rotation=0,  labelsize=6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix_h5.png', dpi=150)
plt.show()

# 6c. Confusion Matrix TFLite
cm_tflite_norm = confusion_matrix(y, y_pred_tflite).astype(float)
cm_tflite_norm /= cm_tflite_norm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm_tflite_norm, ax=ax, cmap='Oranges', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.2, annot=False)
ax.set_title('Confusion Matrix — TFLite (Normalize)', fontsize=15, fontweight='bold')
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek')
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.tick_params(axis='y', rotation=0,  labelsize=6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix_tflite.png', dpi=150)
plt.show()

# 6d. Sınıf Bazlı F1 — H5
f1_per_class = f1_score(y, y_pred_h5, average=None, zero_division=0)
sorted_idx   = np.argsort(f1_per_class)
colors = ['#d62728' if v<0.5 else '#2ca02c' if v>=0.8 else '#ff7f0e'
          for v in f1_per_class[sorted_idx]]
fig, ax = plt.subplots(figsize=(14, max(8, NUM_CLASSES*0.2)))
ax.barh([str(class_names[i]) for i in sorted_idx],
        f1_per_class[sorted_idx], color=colors, edgecolor='white', height=0.8)
ax.set_xlim(0, 1.05)
ax.axvline(0.5, color='red',   linestyle='--', lw=1, alpha=0.6)
ax.axvline(0.8, color='green', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('F1 Skoru')
ax.set_title('H5 — Sınıf Bazlı F1 Skoru', fontsize=14, fontweight='bold')
patches = [mpatches.Patch(color='#d62728', label='<0.50'),
           mpatches.Patch(color='#ff7f0e', label='0.50–0.80'),
           mpatches.Patch(color='#2ca02c', label='≥0.80')]
ax.legend(handles=patches, loc='lower right')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/f1_per_class_h5.png', dpi=150)
plt.show()

# 6e. En Çok Karıştırılan 15 Çift
cm_h5_raw  = confusion_matrix(y, y_pred_h5)
cm_no_diag = cm_h5_raw.copy(); np.fill_diagonal(cm_no_diag, 0)
top15_idx  = np.argsort(cm_no_diag.flatten())[::-1][:15]
rows = top15_idx // NUM_CLASSES
cols = top15_idx  % NUM_CLASSES
pairs  = [f"{class_names[r]} → {class_names[c]}" for r,c in zip(rows,cols)]
counts = [cm_no_diag[r,c] for r,c in zip(rows,cols)]
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(pairs[::-1], counts[::-1], color='#9467bd', edgecolor='white')
ax.set_xlabel('Yanlış Tahmin Sayısı')
ax.set_title('En Çok Karıştırılan 15 Sınıf Çifti (H5)', fontsize=14, fontweight='bold')
for i,v in enumerate(counts[::-1]):
    ax.text(v+0.1, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/top15_confused_pairs.png', dpi=150)
plt.show()

# 6f. Örnek Sayısı vs Doğruluk
correct   = (y_pred_h5 == y)
class_acc = [correct[y==i].mean()*100 if (y==i).sum()>0 else 0 for i in range(NUM_CLASSES)]
class_cnt = [(y==i).sum() for i in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(class_cnt, class_acc, c=class_acc, cmap='RdYlGn',
                vmin=0, vmax=100, s=80, alpha=0.75, edgecolors='grey', lw=0.5)
plt.colorbar(sc, ax=ax, label='Doğruluk (%)')
ax.set_xlabel('Sınıf Başına Örnek Sayısı')
ax.set_ylabel('Sınıf Doğruluğu (%)')
ax.set_title('Örnek Sayısı vs Doğruluk (H5)', fontsize=14, fontweight='bold')
ax.axhline(50, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/sample_vs_accuracy.png', dpi=150)
plt.show()

# ──────────────────────────────────────────────
# 7. ÖZET
# ──────────────────────────────────────────────
print("\n" + "="*50)
print("         TEST SONUÇ ÖZETİ")
print("="*50)
print(f"  Video (örnek) sayısı : {len(y):,}")
print(f"  Sınıf sayısı         : {NUM_CLASSES}")
print(f"  H5  Accuracy         : {acc_h5*100:.2f}%")
print(f"  TFLite Accuracy      : {acc_tflite*100:.2f}%")
print(f"  H5  F1 (macro)       : {f1_score(y,y_pred_h5,    average='macro',zero_division=0)*100:.2f}%")
print(f"  TFLite F1 (macro)    : {f1_score(y,y_pred_tflite,average='macro',zero_division=0)*100:.2f}%")
print("="*50)

Ham veri shape: (166357, 228)
Koordinat sütun sayısı: 225
Kullanılacak koordinat sütunu: 211
Toplam video (örnek) sayısı: 2808

Sequence array shape : (2808, 60, 211)
Sınıf sayısı         : 226
Toplam örnek         : 2808

── H5 modeli yükleniyor ──


Model giriş şekli: (None, 60, 211)
88/88 ━━━━━━━━━━━━━━━━━━━━ 7s 76ms/step
H5 Doğruluğu: 0.43%

── TFLite modeli yükleniyor ──


RuntimeError: Select TensorFlow op(s), included in the given model, is(are) not supported by this interpreter. Make sure you apply/link the Flex delegate before inference. For the Android, it can be resolved by adding "org.tensorflow:tensorflow-lite-select-tf-ops" dependency. See instructions: https://www.tensorflow.org/lite/guide/ops_selectNode number 1 (FlexTensorListReserve) failed to prepare.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TEST_CSV     = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'
MODEL_H5     = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'

# ──────────────────────────────────────────────
# 1. VERİ YÜKLEME
# ──────────────────────────────────────────────
df = pd.read_csv(TEST_CSV)
print(f"Ham veri shape: {df.shape}")

META_COLS  = ['video_name', 'class_id', 'frame_no']
COORD_COLS = [c for c in df.columns if c not in META_COLS]
print(f"Koordinat sütun sayısı: {len(COORD_COLS)}")

# ──────────────────────────────────────────────
# 2. MODEL GİRİŞ ŞEKLİNE GÖRE SEQUENCE OLUŞTURMA
# ──────────────────────────────────────────────
TIMESTEPS = 60
FEATURES  = 211

USE_COLS = COORD_COLS[:FEATURES]
print(f"Kullanılacak koordinat sütunu: {len(USE_COLS)}")

sequences = []
labels    = []

grouped = df.groupby(['video_name', 'class_id'], sort=False)
print(f"Toplam video (örnek) sayısı: {len(grouped)}")

for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    frames = group[USE_COLS].values.astype(np.float32)
    n = len(frames)

    if n >= TIMESTEPS:
        start = (n - TIMESTEPS) // 2
        seq   = frames[start:start + TIMESTEPS]
    else:
        pad = np.zeros((TIMESTEPS - n, FEATURES), dtype=np.float32)
        seq = np.vstack([frames, pad])

    sequences.append(seq)
    labels.append(class_id)

X = np.array(sequences, dtype=np.float32)   # (n_videos, 60, 211)
y_raw = np.array(labels)

# -------------------------------------------------------------
# DİKKAT: %0.43 DOĞRULUK HATASININ ÇÖZÜM NOKTASI
# -------------------------------------------------------------
# Eğer CSV dosyasındaki 'class_id' sütunu zaten 0, 1, 2 gibi SAYILARDAN oluşuyorsa:
# Eğitimde de bu sayıları kullandıysan LabelEncoder KULLANMAMALISIN.
if str(y_raw[0]).isdigit():
    print("\n[BİLGİ] Sınıflar sayısal tespit edildi. Doğrudan kullanılıyor.")
    y = y_raw.astype(int)
    class_names = np.unique(y)
else:
    # Eğer 'class_id' kelimelerden oluşuyorsa (örn: 'merhaba'):
    print("\n[UYARI] Sınıflar metin formatında. LabelEncoder uygulanıyor.")
    print("NOT: Eğitimdeki numaralandırma sırası ile bu testteki sıra uyuşmazsa accuracy %0 çıkabilir!")
    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    class_names = le.classes_

NUM_CLASSES = len(class_names)

print(f"\nSequence array shape : {X.shape}")
print(f"Sınıf sayısı         : {NUM_CLASSES}")
print(f"Toplam örnek         : {len(y)}")

# ──────────────────────────────────────────────
# 3. H5 TAHMİN
# ──────────────────────────────────────────────
print("\n── H5 modeli yükleniyor ──")
model_h5 = tf.keras.models.load_model(MODEL_H5)

probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)
acc_h5    = accuracy_score(y, y_pred_h5)
print(f"\n>>> H5 MODEL DOĞRULUĞU (ACCURACY): %{acc_h5*100:.2f} <<<\n")

# ──────────────────────────────────────────────
# 4. GÖRSEL ÇIKTILAR VE ÖZET (Sadece H5 İçin)
# ──────────────────────────────────────────────
print("Görseller kaydediliyor...")

# Confusion Matrix H5
cm_h5_norm = confusion_matrix(y, y_pred_h5).astype(float)
# ZeroDivision hatasını önlemek için güvenli bölme
row_sums = cm_h5_norm.sum(axis=1, keepdims=True)
cm_h5_norm = np.divide(cm_h5_norm, row_sums, out=np.zeros_like(cm_h5_norm), where=row_sums!=0)

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm_h5_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.2, annot=False)
ax.set_title('Karmaşıklık Matrisi — H5', fontsize=15, fontweight='bold')
ax.set_xlabel('Tahmin Edilen Sınıf')
ax.set_ylabel('Gerçek Sınıf')
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.tick_params(axis='y', rotation=0,  labelsize=6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix_h5.png', dpi=150)
plt.close()

# En Çok Karıştırılan 15 Çift
cm_h5_raw  = confusion_matrix(y, y_pred_h5)
cm_no_diag = cm_h5_raw.copy()
np.fill_diagonal(cm_no_diag, 0)
top15_idx  = np.argsort(cm_no_diag.flatten())[::-1][:15]
rows = top15_idx // NUM_CLASSES
cols = top15_idx  % NUM_CLASSES
pairs  = [f"{class_names[r]} → {class_names[c]}" for r,c in zip(rows,cols)]
counts = [cm_no_diag[r,c] for r,c in zip(rows,cols)]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(pairs[::-1], counts[::-1], color='#9467bd', edgecolor='white')
ax.set_xlabel('Yanlış Tahmin Sayısı')
ax.set_title('En Çok Karıştırılan 15 Sınıf Çifti', fontsize=14, fontweight='bold')
for i,v in enumerate(counts[::-1]):
    ax.text(v+0.1, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/top15_confused_pairs.png', dpi=150)
plt.close()

print("\n" + "="*50)
print("        TEST SONUÇ ÖZETİ (H5)")
print("="*50)
print(f" Video (örnek) sayısı : {len(y)}")
print(f" Sınıf sayısı         : {NUM_CLASSES}")
print(f" Accuracy             : {acc_h5*100:.2f}%")
print(f" F1 Skoru (Macro)     : {f1_score(y,y_pred_h5, average='macro',zero_division=0)*100:.2f}%")
print("="*50)
print("Drive'a kaydedilen görseller:")
print("- confusion_matrix_h5.png")
print("- top15_confused_pairs.png")

Ham veri shape: (166357, 228)
Koordinat sütun sayısı: 225
Kullanılacak koordinat sütunu: 211
Toplam video (örnek) sayısı: 2808



[BİLGİ] Sınıflar sayısal tespit edildi. Doğrudan kullanılıyor.

Sequence array shape : (2808, 60, 211)
Sınıf sayısı         : 226
Toplam örnek         : 2808

── H5 modeli yükleniyor ──
88/88 ━━━━━━━━━━━━━━━━━━━━ 7s 76ms/step

>>> H5 MODEL DOĞRULUĞU (ACCURACY): %0.43 <<<

Görseller kaydediliyor...

        TEST SONUÇ ÖZETİ (H5)
 Video (örnek) sayısı : 2808
 Sınıf sayısı         : 226
 Accuracy             : 0.43%
 F1 Skoru (Macro)     : 0.18%
Drive'a kaydedilen görseller:
- confusion_matrix_h5.png
- top15_confused_pairs.png


In [ ]:
import math

# --- Java'daki İndeksler ---
IDX_NOSE_X  = 0;  IDX_NOSE_Y  = 1
IDX_RH_X    = 99; IDX_RH_Y    = 100
IDX_LH_X    = 162; IDX_LH_Y   = 163
IDX_MOUTH_X = 27; IDX_MOUTH_Y = 28
IDX_REYE_X  = 6;  IDX_REYE_Y  = 7
IDX_LEYE_X  = 15; IDX_LEYE_Y  = 16
IDX_RS_X    = 33; IDX_RS_Y    = 34
IDX_LS_X    = 36; IDX_LS_Y    = 37

def dist(x1, y1, x2, y2):
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

# Java'daki extractFeatures metodunun Python karşılığı
def extract_features(raw_frame, prev_rh, prev_lh):
    nx = raw_frame[IDX_NOSE_X]
    ny = raw_frame[IDX_NOSE_Y]

    base = []
    # Pose: İlk 25 landmark (75 değer)
    for i in range(25):
        base.append((raw_frame[i*3] - nx) / 2.0)
        base.append((raw_frame[i*3+1] - ny) / 2.0)
        base.append(raw_frame[i*3+2] / 2.0)

    # Sağ El: 21 landmark (63 değer)
    for i in range(21):
        base.append((raw_frame[99+i*3] - nx) / 2.0)
        base.append((raw_frame[99+i*3+1] - ny) / 2.0)
        base.append(raw_frame[99+i*3+2] / 2.0)

    # Sol El: 21 landmark (63 değer)
    for i in range(21):
        base.append((raw_frame[162+i*3] - nx) / 2.0)
        base.append((raw_frame[162+i*3+1] - ny) / 2.0)
        base.append(raw_frame[162+i*3+2] / 2.0)

    rhX = raw_frame[IDX_RH_X]; rhY = raw_frame[IDX_RH_Y]
    lhX = raw_frame[IDX_LH_X]; lhY = raw_frame[IDX_LH_Y]

    extra = [0.0] * 10
    extra[0] = dist(rhX, rhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y])
    extra[1] = dist(rhX, rhY, raw_frame[IDX_REYE_X],  raw_frame[IDX_REYE_Y])
    extra[2] = dist(rhX, rhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y])
    extra[3] = dist(rhX, rhY, raw_frame[IDX_RS_X],    raw_frame[IDX_RS_Y])
    extra[4] = dist(lhX, lhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y])
    extra[5] = dist(lhX, lhY, raw_frame[IDX_LEYE_X],  raw_frame[IDX_LEYE_Y])
    extra[6] = dist(lhX, lhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y])
    extra[7] = dist(lhX, lhY, raw_frame[IDX_LS_X],    raw_frame[IDX_LS_Y])
    extra[8] = dist(rhX, rhY, prev_rh[0], prev_rh[1])
    extra[9] = dist(lhX, lhY, prev_lh[0], prev_lh[1])

    return np.array(base + extra, dtype=np.float32), (rhX, rhY), (lhX, lhY)

# -------------------------------------------------------------
# 2. MODEL GİRİŞ ŞEKLİNE GÖRE SEQUENCE OLUŞTURMA (GÜNCELLENDİ)
# -------------------------------------------------------------
TIMESTEPS = 60
FEATURES  = 211

sequences = []
labels    = []

grouped = df.groupby(['video_name', 'class_id'], sort=False)
print(f"Toplam video (örnek) sayısı: {len(grouped)}")

for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    raw_frames = group[COORD_COLS].values.astype(np.float32) # Tamamı alınıyor (225)
    n = len(raw_frames)

    # Videonun ortasından 60 kare al, yetmezse başına sıfır ekle
    if n >= TIMESTEPS:
        start = (n - TIMESTEPS) // 2
        selected_raw_frames = raw_frames[start:start + TIMESTEPS]
    else:
        pad_size = TIMESTEPS - n
        pad_raw = np.zeros((pad_size, len(COORD_COLS)), dtype=np.float32)
        selected_raw_frames = np.vstack([pad_raw, raw_frames])

    processed_seq = []
    # Her video başında önceki el koordinatları sıfırlanır
    prev_rh = (0.0, 0.0)
    prev_lh = (0.0, 0.0)

    for raw_frame in selected_raw_frames:
        # Eğer kare tamamen sıfırsa (padding), sonucu da sıfır yap
        if np.all(raw_frame == 0):
            processed_seq.append(np.zeros(FEATURES, dtype=np.float32))
        else:
            feat, prev_rh, prev_lh = extract_features(raw_frame, prev_rh, prev_lh)
            processed_seq.append(feat)

    sequences.append(processed_seq)
    labels.append(class_id)

X = np.array(sequences, dtype=np.float32)
y_raw = np.array(labels)

# Etiketleri sayısal hale getirme (önceki uyarımız geçerli)
if str(y_raw[0]).isdigit():
    y = y_raw.astype(int)
    class_names = np.unique(y)
else:
    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    class_names = le.classes_

NUM_CLASSES = len(class_names)
print(f"\nSequence array shape : {X.shape}")

Toplam video (örnek) sayısı: 2808

Sequence array shape : (2808, 60, 211)


In [ ]:
from sklearn.metrics import accuracy_score

print("Güncellenmiş veri formatıyla tahminler yapılıyor...")
model_h5 = tf.keras.models.load_model(MODEL_H5)

# Tahmin işlemini başlat (batch_size=32 olduğu için yine 88 adım göreceksin)
probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)

acc_h5 = accuracy_score(y, y_pred_h5)
print(f"\n>>> GÜNCEL H5 MODEL DOĞRULUĞU (ACCURACY): %{acc_h5*100:.2f} <<<\n")

Güncellenmiş veri formatıyla tahminler yapılıyor...


88/88 ━━━━━━━━━━━━━━━━━━━━ 13s 137ms/step

>>> GÜNCEL H5 MODEL DOĞRULUĞU (ACCURACY): %56.80 <<<



In [ ]:
# Teşhis kodu — modeli yeniden eğitmeden önce çalıştır
print("=== SÜTUN SIRALAMA TEŞHİSİ ===")
print(f"\nCSV'deki COORD_COLS sayısı : {len(COORD_COLS)}")
print(f"İlk 10 sütun : {COORD_COLS[:10]}")
print(f"Son 10 sütun : {COORD_COLS[-10:]}")

print("\n=== HAM VERİ İSTATİSTİKLERİ (CSV) ===")
sample_group = list(df.groupby(['video_name','class_id']))[0]
sample_raw   = sample_group[1].sort_values('frame_no')[COORD_COLS].values.astype(np.float32)
print(f"Örnek frame shape : {sample_raw.shape}")
print(f"Min / Max / Mean  : {sample_raw.min():.4f} / {sample_raw.max():.4f} / {sample_raw.mean():.4f}")

print("\n=== extract_features ÇIKTISI ===")
prev_rh = (0.0, 0.0); prev_lh = (0.0, 0.0)
feat, _, _ = extract_features(sample_raw[0], prev_rh, prev_lh)
print(f"Feature vektör uzunluğu : {len(feat)}")
print(f"Min / Max / Mean        : {feat.min():.4f} / {feat.max():.4f} / {feat.mean():.4f}")
print(f"İlk 10 değer            : {feat[:10]}")

print("\n=== MODEL ÇIKTISINDAN ÖRNEK ===")
test_input = X[:5]  # 5 örnek
preds = model_h5.predict(test_input, verbose=0)
print(f"Tahmin olasılıkları (max): {preds.max(axis=1)}")   # yüksek olmalı
print(f"Tahmin edilen sınıflar   : {np.argmax(preds, axis=1)}")
print(f"Gerçek sınıflar          : {y[:5]}")

print("\n=== CSV'DE SÜTUN İSİMLERİ pose_0_x GİBİ Mİ? ===")
# CSV'deki ham değerler zaten normalize mi yoksa piksel mi?
print(f"pose_0_x ilk 5 değer: {df['pose_0_x'].values[:5]}")
print(f"pose_0_y ilk 5 değer: {df['pose_0_y'].values[:5]}")

# extract_features içinde raw_frame[0] = pose_0_x mi?
# CSV sütun sırası ile Java indeksleri örtüşüyor mu kontrol et:
print(f"\nCSV COORD_COLS[0] (pose_0_x) = raw_frame[0] → IDX_NOSE_X=0 ✓ mü?")
print(f"COORD_COLS[0] = {COORD_COLS[0]}")
print(f"COORD_COLS[1] = {COORD_COLS[1]}")
print(f"COORD_COLS[2] = {COORD_COLS[2]}")
# Beklenen: pose_0_x, pose_0_y, pose_0_z

=== SÜTUN SIRALAMA TEŞHİSİ ===

CSV'deki COORD_COLS sayısı : 225
İlk 10 sütun : ['pose_0_x', 'pose_0_y', 'pose_0_z', 'pose_1_x', 'pose_1_y', 'pose_1_z', 'pose_2_x', 'pose_2_y', 'pose_2_z', 'pose_3_x']
Son 10 sütun : ['lh_17_z', 'lh_18_x', 'lh_18_y', 'lh_18_z', 'lh_19_x', 'lh_19_y', 'lh_19_z', 'lh_20_x', 'lh_20_y', 'lh_20_z']

=== HAM VERİ İSTATİSTİKLERİ (CSV) ===
Örnek frame shape : (61, 225)
Min / Max / Mean  : -0.9549 / 1.2989 / 0.2906

=== extract_features ÇIKTISI ===
Feature vektör uzunluğu : 211
Min / Max / Mean        : -0.2194 / 1.0108 / 0.0763
İlk 10 değer            : [ 0.          0.         -0.21943066  0.00535613 -0.01023852 -0.2114217
  0.00915536 -0.00980381 -0.21151242  0.01284617]

=== MODEL ÇIKTISINDAN ÖRNEK ===
Tahmin olasılıkları (max): [0.9857941  0.99844897 0.9074623  0.7625195  0.9999989 ]
Tahmin edilen sınıflar   : [133  61   8 169  77]
Gerçek sınıflar          : [133  61  32 169  77]

=== CSV'DE SÜTUN İSİMLERİ pose_0_x GİBİ Mİ? ===
pose_0_x ilk 5 değer: [0.47122

In [ ]:
# ── GERÇEK DOĞRULUĞU HESAPLA ──
print("=== ETİKET KONTROLÜ ===")
print(f"y_raw (ham class_id) ilk 10  : {y_raw[:10]}")
print(f"y (encode sonrası) ilk 10    : {y[:10]}")
print(f"class_names ilk 10           : {class_names[:10]}")
print(f"class_names[-10:]            : {class_names[-10:]}")
print(f"NUM_CLASSES                  : {NUM_CLASSES}")
print(f"Model çıkış nöron sayısı     : {model_h5.output_shape[-1]}")

# Model çıkışı 225 mi 226 mı?
print(f"\nprobs_h5 shape               : {probs_h5.shape}")

# Tahmin edilen sınıf indeksleri ile gerçek etiketler aynı uzayda mı?
print(f"\ny max değeri                 : {y.max()}")
print(f"y_pred_h5 max değeri         : {y_pred_h5.max()}")
print(f"y min değeri                 : {y.min()}")
print(f"y_pred_h5 min değeri         : {y_pred_h5.min()}")

# Doğrudan class_id karşılaştırması (encode olmadan)
# Model çıktısı zaten class_id'ye karşılık geliyorsa:
y_pred_classid = class_names[y_pred_h5]  # tahmin edilen class_id'ler
correct_direct = (y_pred_classid == y_raw)
acc_direct = correct_direct.mean()
print(f"\nDirekt class_id karşılaştırma accuracy : %{acc_direct*100:.2f}")
print(f"LabelEncoder'lı accuracy              : %{accuracy_score(y, y_pred_h5)*100:.2f}")

=== ETİKET KONTROLÜ ===
y_raw (ham class_id) ilk 10  : [133  61  32 169  77 131 114 143 135 194]
y (encode sonrası) ilk 10    : [133  61  32 169  77 131 114 143 135 194]
class_names ilk 10           : [0 1 2 3 4 5 6 7 8 9]
class_names[-10:]            : [216 217 218 219 220 221 222 223 224 225]
NUM_CLASSES                  : 226
Model çıkış nöron sayısı     : 226

probs_h5 shape               : (2808, 226)

y max değeri                 : 225
y_pred_h5 max değeri         : 224
y min değeri                 : 0
y_pred_h5 min değeri         : 0

Direkt class_id karşılaştırma accuracy : %56.80
LabelEncoder'lı accuracy              : %56.80


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

TEST_CSV     = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'
MODEL_H5     = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'
MODEL_TFLITE = '/content/drive/MyDrive/isaret_dili_beyni_v2_android.tflite'

# ──────────────────────────────────────────────
# 1. SÜTUN İSİMLERİNİ EĞİTİMLE AYNI ŞEKİLDE HAZIRLA
# ──────────────────────────────────────────────
df = pd.read_csv(TEST_CSV)
print(f"Ham veri shape: {df.shape}")

META_COLS     = ['video_name', 'class_id', 'frame_no']
feature_names = [c for c in df.columns if c not in META_COLS]  # eğitimdeki header[3:] ile aynı

# Eğitimde pose_25+ sütunları atılıyordu
bad_indices = set()
for i, col_name in enumerate(feature_names):
    if 'pose_' in col_name:
        try:
            if int(col_name.split('_')[1]) >= 25:
                bad_indices.add(i)
        except:
            pass

valid_indices = [i for i in range(len(feature_names)) if i not in bad_indices]
print(f"Toplam özellik   : {len(feature_names)}")
print(f"Geçerli indeks   : {len(valid_indices)}  (pose_25+ atıldı)")

# Eğitimdeki idx hesapları
def get_idx(name):
    try: return feature_names.index(name)
    except: return -1

idx_nose_x  = get_idx('pose_0_x');  idx_nose_y  = get_idx('pose_0_y')
idx_rh_x    = get_idx('rh_0_x');    idx_rh_y    = get_idx('rh_0_y')
idx_lh_x    = get_idx('lh_0_x');    idx_lh_y    = get_idx('lh_0_y')
idx_mouth_x = get_idx('pose_9_x');  idx_mouth_y = get_idx('pose_9_y')
idx_reye_x  = get_idx('pose_2_x');  idx_reye_y  = get_idx('pose_2_y')
idx_leye_x  = get_idx('pose_5_x');  idx_leye_y  = get_idx('pose_5_y')
idx_rs_x    = get_idx('pose_11_x'); idx_rs_y    = get_idx('pose_11_y')
idx_ls_x    = get_idx('pose_12_x'); idx_ls_y    = get_idx('pose_12_y')

ADDED_FEATURES   = 10
BASE_FEATURES    = len(valid_indices)
NEW_NUM_FEATURES = BASE_FEATURES + ADDED_FEATURES
NUM_CLASSES      = 226
MAX_FRAMES       = 60

print(f"NEW_NUM_FEATURES : {NEW_NUM_FEATURES}  (modelin beklediği: {MODEL_H5 and 211})")

def dist_raw(raw, ix, iy, jx, jy):
    if any(v < 0 or v >= len(raw) for v in [ix, iy, jx, jy]): return 0.0
    return ((raw[ix]-raw[jx])**2 + (raw[iy]-raw[jy])**2)**0.5

# ──────────────────────────────────────────────
# 2. EĞİTİMLE BİREBİR AYNI ÖN İŞLEME
# ──────────────────────────────────────────────
sequences, labels = [], []
grouped = df.groupby('video_name', sort=False)
print(f"\nToplam video: {len(grouped)}")

for video_name, group in grouped:
    group       = group.sort_values('frame_no')
    class_id    = int(group['class_id'].iloc[0])
    raw_all     = group[feature_names].values.astype(np.float64)

    frames         = []
    prev_rh_x = prev_rh_y = prev_lh_x = prev_lh_y = 0.0

    for raw in raw_all:
        nx = raw[idx_nose_x] if idx_nose_x >= 0 else 0.0
        ny = raw[idx_nose_y] if idx_nose_y >= 0 else 0.0

        norm_feat = []
        for i in valid_indices:
            val = raw[i] if i < len(raw) else 0.0
            if feature_names[i].endswith('_x'): val -= nx
            elif feature_names[i].endswith('_y'): val -= ny
            norm_feat.append(val / 2.0)

        d_rh_mouth = dist_raw(raw, idx_rh_x, idx_rh_y, idx_mouth_x, idx_mouth_y)
        d_rh_reye  = dist_raw(raw, idx_rh_x, idx_rh_y, idx_reye_x,  idx_reye_y)
        d_rh_nose  = dist_raw(raw, idx_rh_x, idx_rh_y, idx_nose_x,  idx_nose_y)
        d_rh_shldr = dist_raw(raw, idx_rh_x, idx_rh_y, idx_rs_x,    idx_rs_y)
        d_lh_mouth = dist_raw(raw, idx_lh_x, idx_lh_y, idx_mouth_x, idx_mouth_y)
        d_lh_leye  = dist_raw(raw, idx_lh_x, idx_lh_y, idx_leye_x,  idx_leye_y)
        d_lh_nose  = dist_raw(raw, idx_lh_x, idx_lh_y, idx_nose_x,  idx_nose_y)
        d_lh_shldr = dist_raw(raw, idx_lh_x, idx_lh_y, idx_ls_x,    idx_ls_y)

        cur_rh_x = raw[idx_rh_x] if idx_rh_x >= 0 else 0.0
        cur_rh_y = raw[idx_rh_y] if idx_rh_y >= 0 else 0.0
        cur_lh_x = raw[idx_lh_x] if idx_lh_x >= 0 else 0.0
        cur_lh_y = raw[idx_lh_y] if idx_lh_y >= 0 else 0.0

        vel_rh = ((cur_rh_x-prev_rh_x)**2 + (cur_rh_y-prev_rh_y)**2)**0.5
        vel_lh = ((cur_lh_x-prev_lh_x)**2 + (cur_lh_y-prev_lh_y)**2)**0.5
        prev_rh_x, prev_rh_y = cur_rh_x, cur_rh_y
        prev_lh_x, prev_lh_y = cur_lh_x, cur_lh_y

        norm_feat.extend([d_rh_mouth, d_rh_reye, d_rh_nose, d_rh_shldr,
                          d_lh_mouth, d_lh_leye, d_lh_nose, d_lh_shldr,
                          vel_rh, vel_lh])

        if len(norm_feat) > NEW_NUM_FEATURES:
            norm_feat = norm_feat[:NEW_NUM_FEATURES]
        elif len(norm_feat) < NEW_NUM_FEATURES:
            norm_feat.extend([0.0] * (NEW_NUM_FEATURES - len(norm_feat)))

        frames.append(norm_feat)

    # Eğitimle aynı: baştan MAX_FRAMES al, yetmezse sona sıfır pad
    if len(frames) > MAX_FRAMES:
        frames = frames[:MAX_FRAMES]
    else:
        frames.extend([[0.0]*NEW_NUM_FEATURES] * (MAX_FRAMES - len(frames)))

    # video_name yerine class_id bazında reset yok — eğitimle aynı
    prev_rh_x = prev_rh_y = prev_lh_x = prev_lh_y = 0.0

    sequences.append(frames)
    labels.append(class_id)

X      = np.array(sequences, dtype=np.float32)
y_true = np.array(labels,    dtype=np.int32)
print(f"X shape  : {X.shape}")
print(f"y shape  : {y_true.shape}")
print(f"Sınıflar : {np.unique(y_true)[:10]} ...")

# ──────────────────────────────────────────────
# 3. H5 TAHMİN
# ──────────────────────────────────────────────
print("\n── H5 modeli yükleniyor ──")
model_h5    = tf.keras.models.load_model(MODEL_H5)
print(f"Giriş şekli: {model_h5.input_shape}")

probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)
acc_h5    = accuracy_score(y_true, y_pred_h5)
print(f"\n>>> H5 ACCURACY: %{acc_h5*100:.2f} <<<")

# ──────────────────────────────────────────────
# 4. TFLite TAHMİN
# ──────────────────────────────────────────────
print("\n── TFLite modeli yükleniyor ──")
interpreter = tf.lite.Interpreter(model_path=MODEL_TFLITE)
interpreter.allocate_tensors()
inp_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()

preds_tflite = []
for i in range(len(X)):
    interpreter.set_tensor(inp_det[0]['index'], X[i:i+1])
    interpreter.invoke()
    preds_tflite.append(np.argmax(interpreter.get_tensor(out_det[0]['index'])))

y_pred_tflite = np.array(preds_tflite)
acc_tflite    = accuracy_score(y_true, y_pred_tflite)
print(f">>> TFLite ACCURACY: %{acc_tflite*100:.2f} <<<")

# ──────────────────────────────────────────────
# 5. RAPORLAR
# ──────────────────────────────────────────────
print("\n══ H5 Sınıflandırma Raporu ══")
print(classification_report(y_true, y_pred_h5,
      labels=np.arange(NUM_CLASSES),
      target_names=[str(i) for i in range(NUM_CLASSES)],
      zero_division=0))

# ──────────────────────────────────────────────
# 6. GÖRSELLER
# ──────────────────────────────────────────────
f1_h5  = f1_score(y_true, y_pred_h5,      average='macro', zero_division=0)
pr_h5  = precision_score(y_true, y_pred_h5, average='macro', zero_division=0)
rc_h5  = recall_score(y_true, y_pred_h5,   average='macro', zero_division=0)
f1_tfl = f1_score(y_true, y_pred_tflite,  average='macro', zero_division=0)

# 6a. Genel Metrik
fig, ax = plt.subplots(figsize=(9, 5))
names = ['Accuracy','F1 (macro)','Precision','Recall']
h5v   = [acc_h5*100, f1_h5*100, pr_h5*100, rc_h5*100]
tflv  = [acc_tflite*100, f1_tfl*100,
         precision_score(y_true,y_pred_tflite,average='macro',zero_division=0)*100,
         recall_score(y_true,y_pred_tflite,average='macro',zero_division=0)*100]
x = np.arange(len(names)); w = 0.35
b1 = ax.bar(x-w/2, h5v,  w, label='H5',     color='#4C72B0')
b2 = ax.bar(x+w/2, tflv, w, label='TFLite', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=12)
ax.set_ylim(0,115); ax.set_ylabel('%'); ax.legend(fontsize=11)
ax.set_title('H5 vs TFLite — Genel Metrikler (226 sınıf)', fontsize=13, fontweight='bold')
ax.axhline(50, color='gray',  linestyle='--', alpha=0.4)
ax.axhline(80, color='green', linestyle='--', alpha=0.4)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1.2,
            f'{b.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/metrik_karsilastirma.png', dpi=150); plt.show()

# 6b. Confusion Matrix
cm      = confusion_matrix(y_true, y_pred_h5, labels=np.arange(NUM_CLASSES))
cm_norm = cm.astype(float) / np.where(cm.sum(1,keepdims=True)==0, 1, cm.sum(1,keepdims=True))
fig, ax = plt.subplots(figsize=(22,18))
sns.heatmap(cm_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
            linewidths=0.15, annot=False,
            xticklabels=np.arange(NUM_CLASSES), yticklabels=np.arange(NUM_CLASSES))
ax.set_title('Confusion Matrix — H5 (Normalize)', fontsize=15, fontweight='bold')
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek')
ax.tick_params(axis='x', rotation=90, labelsize=5)
ax.tick_params(axis='y', rotation=0,  labelsize=5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=150); plt.show()

# 6c. Sınıf bazlı F1
f1_cls     = f1_score(y_true, y_pred_h5, average=None,
                       labels=np.arange(NUM_CLASSES), zero_division=0)
sorted_idx = np.argsort(f1_cls)
bar_colors = ['#d62728' if v<0.5 else '#2ca02c' if v>=0.8 else '#ff7f0e'
               for v in f1_cls[sorted_idx]]
fig, ax = plt.subplots(figsize=(14, 40))
ax.barh([str(i) for i in sorted_idx], f1_cls[sorted_idx],
        color=bar_colors, edgecolor='white', height=0.8)
ax.set_xlim(0,1.05)
ax.axvline(0.5, color='red',   linestyle='--', lw=1, alpha=0.6)
ax.axvline(0.8, color='green', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('F1 Skoru', fontsize=12)
ax.set_title('Sınıf Bazlı F1 Skoru — H5', fontsize=14, fontweight='bold')
patches = [mpatches.Patch(color='#d62728', label='F1 < 0.50'),
           mpatches.Patch(color='#ff7f0e', label='0.50–0.80'),
           mpatches.Patch(color='#2ca02c', label='F1 ≥ 0.80')]
ax.legend(handles=patches, loc='lower right', fontsize=10)
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/f1_sinif_bazli.png', dpi=150); plt.show()

# 6d. F1 Dağılımı Histogram
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(f1_cls, bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(f1_cls.mean(),   color='red',    linestyle='--', lw=2, label=f'Ort: {f1_cls.mean():.3f}')
ax.axvline(np.median(f1_cls), color='orange', linestyle='--', lw=2, label=f'Med: {np.median(f1_cls):.3f}')
ax.set_xlabel('F1 Skoru'); ax.set_ylabel('Sınıf Sayısı')
ax.set_title('F1 Skoru Dağılımı', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
iyi   = (f1_cls>=0.8).sum(); orta = ((f1_cls>=0.5)&(f1_cls<0.8)).sum(); zayif=(f1_cls<0.5).sum()
ax.text(0.02,0.95, f'İyi(≥0.8): {iyi}\nOrta: {orta}\nZayıf(<0.5): {zayif}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/f1_dagilimi.png', dpi=150); plt.show()

# 6e. En çok karıştırılan 20 çift
cm_nd = cm.copy(); np.fill_diagonal(cm_nd, 0)
top20 = np.argsort(cm_nd.flatten())[::-1][:20]
rows  = top20 // NUM_CLASSES; cols = top20 % NUM_CLASSES
pairs  = [f"Sınıf {r} → {c}" for r,c in zip(rows,cols)]
counts = [cm_nd[r,c] for r,c in zip(rows,cols)]
fig, ax = plt.subplots(figsize=(11,7))
ax.barh(pairs[::-1], counts[::-1], color='#9467bd', edgecolor='white')
ax.set_xlabel('Yanlış Tahmin Sayısı')
ax.set_title('En Çok Karıştırılan 20 Sınıf Çifti (H5)', fontsize=14, fontweight='bold')
for i,v in enumerate(counts[::-1]):
    ax.text(v+0.1, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/karistirilan_ciftler.png', dpi=150); plt.show()

# 6f. Örnek sayısı vs doğruluk
correct   = (y_pred_h5 == y_true)
cls_acc   = [correct[y_true==i].mean()*100 if (y_true==i).sum()>0 else 0 for i in range(NUM_CLASSES)]
cls_cnt   = [(y_true==i).sum() for i in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(10,6))
sc = ax.scatter(cls_cnt, cls_acc, c=cls_acc, cmap='RdYlGn', vmin=0, vmax=100,
                s=70, alpha=0.75, edgecolors='grey', lw=0.4)
plt.colorbar(sc, ax=ax, label='Doğruluk (%)')
ax.set_xlabel('Sınıf Başına Örnek Sayısı'); ax.set_ylabel('Sınıf Doğruluğu (%)')
ax.set_title('Örnek Sayısı vs Doğruluk (H5)', fontsize=14, fontweight='bold')
ax.axhline(50, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ornek_vs_dogruluk.png', dpi=150); plt.show()

# ──────────────────────────────────────────────
# 7. ÖZET
# ──────────────────────────────────────────────
print("\n" + "="*52)
print("            TEST SONUÇ ÖZETİ")
print("="*52)
print(f"  Video (örnek) : {len(y_true):,}")
print(f"  Sınıf sayısı  : {NUM_CLASSES}")
print(f"  H5  Accuracy  : %{acc_h5*100:.2f}")
print(f"  TFLite Acc.   : %{acc_tflite*100:.2f}")
print(f"  H5  F1 (macro): %{f1_h5*100:.2f}")
print(f"  TFLite F1     : %{f1_tfl*100:.2f}")
print(f"  İyi sınıf(≥.8): {iyi} / {NUM_CLASSES}")
print(f"  Zayıf sınıf   : {zayif} / {NUM_CLASSES}")
print("="*52)

Ham veri shape: (166357, 228)
Toplam özellik   : 225
Geçerli indeks   : 201  (pose_25+ atıldı)
NEW_NUM_FEATURES : 211  (modelin beklediği: 211)

Toplam video: 2808
X shape  : (2808, 60, 211)
y shape  : (2808,)
Sınıflar : [0 1 2 3 4 5 6 7 8 9] ...

── H5 modeli yükleniyor ──


Giriş şekli: (None, 60, 211)
88/88 ━━━━━━━━━━━━━━━━━━━━ 8s 91ms/step

>>> H5 ACCURACY: %78.38 <<<

── TFLite modeli yükleniyor ──


RuntimeError: Select TensorFlow op(s), included in the given model, is(are) not supported by this interpreter. Make sure you apply/link the Flex delegate before inference. For the Android, it can be resolved by adding "org.tensorflow:tensorflow-lite-select-tf-ops" dependency. See instructions: https://www.tensorflow.org/lite/guide/ops_selectNode number 1 (FlexTensorListReserve) failed to prepare.

In [ ]:
for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    raw_frames = group[COORD_COLS].values.astype(np.float32)
    n = len(raw_frames)

    # ---------------------------------------------------------
    # DÜZELTME BURADA: Eğitim koduyla birebir aynı kesme/doldurma
    # ---------------------------------------------------------
    if n >= TIMESTEPS:
        # Ortadan değil, doğrudan BAŞTAN ilk 60 kareyi alıyoruz
        selected_raw_frames = raw_frames[:TIMESTEPS]
    else:
        # Eksik kareleri BAŞA değil, SONA sıfır olarak ekliyoruz (Post-padding)
        pad_size = TIMESTEPS - n
        pad_raw = np.zeros((pad_size, len(COORD_COLS)), dtype=np.float32)
        selected_raw_frames = np.vstack([raw_frames, pad_raw])

    processed_seq = []
    prev_rh = (0.0, 0.0)
    prev_lh = (0.0, 0.0)

    for raw_frame in selected_raw_frames:
        if np.all(raw_frame == 0):
            processed_seq.append(np.zeros(FEATURES, dtype=np.float32))
        else:
            feat, prev_rh, prev_lh = extract_features(raw_frame, prev_rh, prev_lh)
            processed_seq.append(feat)

    sequences.append(processed_seq)
    labels.append(class_id)

ValueError: too many values to unpack (expected 2)

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TEST_CSV     = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'
MODEL_H5     = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'

# ──────────────────────────────────────────────
# 1. JAVA EXTRACT FEATURES MANTIĞI (PYTHON ÇEVİRİSİ)
# ──────────────────────────────────────────────
IDX_NOSE_X  = 0;  IDX_NOSE_Y  = 1
IDX_RH_X    = 99; IDX_RH_Y    = 100
IDX_LH_X    = 162; IDX_LH_Y   = 163
IDX_MOUTH_X = 27; IDX_MOUTH_Y = 28
IDX_REYE_X  = 6;  IDX_REYE_Y  = 7
IDX_LEYE_X  = 15; IDX_LEYE_Y  = 16
IDX_RS_X    = 33; IDX_RS_Y    = 34
IDX_LS_X    = 36; IDX_LS_Y    = 37

def dist(x1, y1, x2, y2):
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def extract_features(raw_frame, prev_rh, prev_lh):
    nx = raw_frame[IDX_NOSE_X]
    ny = raw_frame[IDX_NOSE_Y]

    base = []
    # Pose (25), Sağ El (21), Sol El (21)
    for i in range(25):
        base.extend([(raw_frame[i*3]-nx)/2.0, (raw_frame[i*3+1]-ny)/2.0, raw_frame[i*3+2]/2.0])
    for i in range(21):
        base.extend([(raw_frame[99+i*3]-nx)/2.0, (raw_frame[99+i*3+1]-ny)/2.0, raw_frame[99+i*3+2]/2.0])
    for i in range(21):
        base.extend([(raw_frame[162+i*3]-nx)/2.0, (raw_frame[162+i*3+1]-ny)/2.0, raw_frame[162+i*3+2]/2.0])

    rhX, rhY = raw_frame[IDX_RH_X], raw_frame[IDX_RH_Y]
    lhX, lhY = raw_frame[IDX_LH_X], raw_frame[IDX_LH_Y]

    extra = [
        dist(rhX, rhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y]),
        dist(rhX, rhY, raw_frame[IDX_REYE_X],  raw_frame[IDX_REYE_Y]),
        dist(rhX, rhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y]),
        dist(rhX, rhY, raw_frame[IDX_RS_X],    raw_frame[IDX_RS_Y]),
        dist(lhX, lhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y]),
        dist(lhX, lhY, raw_frame[IDX_LEYE_X],  raw_frame[IDX_LEYE_Y]),
        dist(lhX, lhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y]),
        dist(lhX, lhY, raw_frame[IDX_LS_X],    raw_frame[IDX_LS_Y]),
        dist(rhX, rhY, prev_rh[0], prev_rh[1]),
        dist(lhX, lhY, prev_lh[0], prev_lh[1])
    ]
    return np.array(base + extra, dtype=np.float32), (rhX, rhY), (lhX, lhY)

# ──────────────────────────────────────────────
# 2. VERİ YÜKLEME VE SEQUENCE OLUŞTURMA
# ──────────────────────────────────────────────
df = pd.read_csv(TEST_CSV)
print(f"Ham veri shape: {df.shape}")

META_COLS  = ['video_name', 'class_id', 'frame_no']
COORD_COLS = [c for c in df.columns if c not in META_COLS]

TIMESTEPS = 60
FEATURES  = 211
sequences, labels = [], []

grouped = df.groupby(['video_name', 'class_id'], sort=False)
print(f"Toplam video (örnek) sayısı: {len(grouped)}")

for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    raw_frames = group[COORD_COLS].values.astype(np.float32)
    n = len(raw_frames)

    # EĞİTİMDEKİ GİBİ: Baştan al, eksikse sona sıfır (Post-padding) ekle
    if n >= TIMESTEPS:
        selected_raw_frames = raw_frames[:TIMESTEPS]
    else:
        pad_size = TIMESTEPS - n
        pad_raw = np.zeros((pad_size, len(COORD_COLS)), dtype=np.float32)
        selected_raw_frames = np.vstack([raw_frames, pad_raw])

    processed_seq = []
    prev_rh, prev_lh = (0.0, 0.0), (0.0, 0.0)

    for raw_frame in selected_raw_frames:
        if np.all(raw_frame == 0):
            processed_seq.append(np.zeros(FEATURES, dtype=np.float32))
        else:
            feat, prev_rh, prev_lh = extract_features(raw_frame, prev_rh, prev_lh)
            processed_seq.append(feat)

    sequences.append(processed_seq)
    labels.append(class_id)

X = np.array(sequences, dtype=np.float32)
y_raw = np.array(labels)

# Etiketleri sayısal hale getirme (Kaymayı önler)
if str(y_raw[0]).isdigit():
    y = y_raw.astype(int)
    class_names = np.unique(y)
else:
    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    class_names = le.classes_

NUM_CLASSES = len(class_names)
print(f"\nSequence array shape : {X.shape}")
print(f"Sınıf sayısı         : {NUM_CLASSES}")

# ──────────────────────────────────────────────
# 3. H5 TAHMİN (PREDICTION)
# ──────────────────────────────────────────────
print("\n── H5 modeli yükleniyor ──")
model_h5 = tf.keras.models.load_model(MODEL_H5)

probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)
acc_h5    = accuracy_score(y, y_pred_h5)
print(f"\n>>> H5 Doğruluğu: {acc_h5*100:.2f}% <<<")

# ──────────────────────────────────────────────
# 4. GÖRSEL ÇIKTILAR (TFLite kısımları temizlendi)
# ──────────────────────────────────────────────
print("\nGrafikler hazırlanıyor ve Drive'a kaydediliyor...")

# 4a. Genel Metrikler (Bar Chart)
metrics = {
    'Accuracy':  acc_h5 * 100,
    'F1 (macro)': f1_score(y, y_pred_h5, average='macro', zero_division=0) * 100,
    'Precision': precision_score(y, y_pred_h5, average='macro', zero_division=0) * 100,
    'Recall':    recall_score(y, y_pred_h5, average='macro', zero_division=0) * 100,
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics.keys(), metrics.values(), color='#4C72B0', edgecolor='white', width=0.5)
ax.set_ylim(0, 110); ax.set_ylabel('%', fontsize=12)
ax.set_title('H5 Model — Genel Performans Metrikleri', fontsize=14, fontweight='bold')
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f'{b.get_height():.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/metric_h5.png', dpi=150)
plt.close()

# 4b. Confusion Matrix (H5)
cm_h5_raw = confusion_matrix(y, y_pred_h5)
row_sums = cm_h5_raw.sum(axis=1, keepdims=True)
cm_h5_norm = np.divide(cm_h5_raw.astype(float), row_sums, out=np.zeros_like(cm_h5_raw, dtype=float), where=row_sums!=0)

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(cm_h5_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.2, annot=False)
ax.set_title('Karmaşıklık Matrisi — H5 (Normalize)', fontsize=16, fontweight='bold')
ax.set_xlabel('Tahmin Edilen Sınıf', fontsize=12)
ax.set_ylabel('Gerçek Sınıf', fontsize=12)
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.tick_params(axis='y', rotation=0,  labelsize=6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix_h5.png', dpi=150)
plt.close()

# 4c. Sınıf Bazlı F1 Skoru
f1_per_class = f1_score(y, y_pred_h5, average=None, zero_division=0)
sorted_idx   = np.argsort(f1_per_class)
colors = ['#d62728' if v<0.5 else '#2ca02c' if v>=0.8 else '#ff7f0e' for v in f1_per_class[sorted_idx]]

fig, ax = plt.subplots(figsize=(14, max(8, NUM_CLASSES*0.2)))
ax.barh([str(class_names[i]) for i in sorted_idx],
        f1_per_class[sorted_idx], color=colors, edgecolor='white', height=0.8)
ax.set_xlim(0, 1.05)
ax.axvline(0.5, color='red',   linestyle='--', lw=1, alpha=0.6)
ax.axvline(0.8, color='green', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('F1 Skoru', fontsize=12)
ax.set_title('H5 — Sınıf Bazlı F1 Skoru (Küçükten Büyüğe)', fontsize=14, fontweight='bold')
patches = [mpatches.Patch(color='#d62728', label='<0.50 (Zayıf)'),
           mpatches.Patch(color='#ff7f0e', label='0.50–0.80 (Orta)'),
           mpatches.Patch(color='#2ca02c', label='≥0.80 (İyi)')]
ax.legend(handles=patches, loc='lower right', fontsize=10)
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/f1_per_class_h5.png', dpi=150)
plt.close()

# 4d. En Çok Karıştırılan 15 Çift
cm_no_diag = cm_h5_raw.copy()
np.fill_diagonal(cm_no_diag, 0)
top15_idx  = np.argsort(cm_no_diag.flatten())[::-1][:15]
rows = top15_idx // NUM_CLASSES
cols = top15_idx  % NUM_CLASSES
pairs  = [f"{class_names[r]} → {class_names[c]}" for r,c in zip(rows,cols)]
counts = [cm_no_diag[r,c] for r,c in zip(rows,cols)]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(pairs[::-1], counts[::-1], color='#9467bd', edgecolor='white')
ax.set_xlabel('Yanlış Tahmin Sayısı', fontsize=12)
ax.set_title('En Çok Karıştırılan 15 Sınıf Çifti (H5)', fontsize=14, fontweight='bold')
for i,v in enumerate(counts[::-1]):
    ax.text(v+0.1, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/top15_confused_pairs.png', dpi=150)
plt.close()

# 4e. Örnek Sayısı vs Doğruluk
correct   = (y_pred_h5 == y)
class_acc = [correct[y==i].mean()*100 if (y==i).sum()>0 else 0 for i in range(NUM_CLASSES)]
class_cnt = [(y==i).sum() for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(class_cnt, class_acc, c=class_acc, cmap='RdYlGn',
                vmin=0, vmax=100, s=80, alpha=0.8, edgecolors='grey', lw=0.5)
plt.colorbar(sc, ax=ax, label='Doğruluk (%)')
ax.set_xlabel('Sınıf Başına Test Örneği Sayısı', fontsize=12)
ax.set_ylabel('Sınıf Doğruluğu (%)', fontsize=12)
ax.set_title('Örnek Sayısı vs Doğruluk (H5)', fontsize=14, fontweight='bold')
ax.axhline(50, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/sample_vs_accuracy.png', dpi=150)
plt.close()

# ──────────────────────────────────────────────
# 5. ÖZET
# ──────────────────────────────────────────────
print("\n" + "="*50)
print("        TEST SONUÇ ÖZETİ (H5 MODELİ)")
print("="*50)
print(f" Video (örnek) sayısı : {len(y):,}")
print(f" Sınıf sayısı         : {NUM_CLASSES}")
print(f" Accuracy             : {metrics['Accuracy']:.2f}%")
print(f" F1 (macro)           : {metrics['F1 (macro)']:.2f}%")
print(f" Precision (macro)    : {metrics['Precision']:.2f}%")
print(f" Recall (macro)       : {metrics['Recall']:.2f}%")
print("="*50)
print("\n💾 Başarıyla Kaydedilen Görseller:")
print(" 1. metric_h5.png")
print(" 2. confusion_matrix_h5.png")
print(" 3. f1_per_class_h5.png")
print(" 4. top15_confused_pairs.png")
print(" 5. sample_vs_accuracy.png")

Ham veri shape: (166357, 228)
Toplam video (örnek) sayısı: 2808

Sequence array shape : (2808, 60, 211)
Sınıf sayısı         : 226

── H5 modeli yükleniyor ──


88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

>>> H5 Doğruluğu: 78.42% <<<

Grafikler hazırlanıyor ve Drive'a kaydediliyor...

        TEST SONUÇ ÖZETİ (H5 MODELİ)
 Video (örnek) sayısı : 2,808
 Sınıf sayısı         : 226
 Accuracy             : 78.42%
 F1 (macro)           : 78.00%
 Precision (macro)    : 79.97%
 Recall (macro)       : 78.36%

💾 Başarıyla Kaydedilen Görseller:
 1. metric_h5.png
 2. confusion_matrix_h5.png
 3. f1_per_class_h5.png
 4. top15_confused_pairs.png
 5. sample_vs_accuracy.png
